# 01. 레거시 기준선 (개선 전)

이 노트북은 데모를 보는 사람이 **자기 시스템의 개선 전 상태**로 보는 레거시 기준선입니다. `00` 노트북과 동일한 결정론적 합성 데이터로 대표 레거시 모델(`Mean`, `Weather`, `ForecastWeather`, `Ldaps`, `SPOT`)을 재구성해 평가합니다.

핵심은 예측 시점 입력만 사용하는 `SPOT`입니다. 이후 `02`(수동 스킬 적용)와 `03`(자동 연구)은 여기서 얻은 `SPOT` 기준선을 넘어서는지, 그리고 안전 게이트를 지키는지를 보여줍니다.

- 저장소 루트에서 실행하세요.
- 산출물은 `artifacts/demo/`에 저장되며, `02` 노트북이 같은 데이터셋을 입력으로 사용합니다.
- `Weather`는 실제 관측치를 쓰는 사후 분석용 오라클이므로 운영 예측 모델이 아닙니다.

In [ ]:
from pathlib import Path

import pandas as pd

from power_forecasting.cli import run_generate_data, run_legacy

ARTIFACT_DIR = Path("artifacts/demo")
dataset_path = run_generate_data(ARTIFACT_DIR, days=60, plants=3, seed=42)
legacy_results = run_legacy(ARTIFACT_DIR, dataset=dataset_path, folds=5)

legacy_metrics = (
    pd.DataFrame(
        {
            "model": name,
            "MAE": result.metrics["MAE"],
            "RMSE": result.metrics["RMSE"],
            "NMAE": result.metrics["NMAE"],
        }
        for name, result in legacy_results.items()
    )
    .sort_values("NMAE")
    .reset_index(drop=True)
)
legacy_metrics

In [ ]:
# SPOT을 예측 시점 기준선으로 고정합니다. 이후 노트북은 이 값을 넘어서야 승격됩니다.
spot_nmae = legacy_metrics.loc[legacy_metrics["model"] == "SPOT", "NMAE"].iat[0]
print(f"개선 전 SPOT 기준선 NMAE: {spot_nmae:.6f}")
print(f"데이터셋 경로: {dataset_path}")
print("이 데이터셋과 SPOT 기준선이 02(수동)와 03(자동)의 출발점입니다.")

## 다음 단계

여기까지가 개선 전 레거시입니다. 같은 데이터셋과 `SPOT` 기준선을 놓고 두 가지 개선 경로를 보여줍니다.

- `02_manual_skill_path.ipynb`: 사람이 통제하는 기본 경로 — `legacy-intake -> AIDM experiment -> AIDD promotion -> human review`.
- `03_auto_research_path.ipynb`: 선택적 자동 Stage 1 연구 경로 — `diagnosis -> bounded proposal -> AIDM -> evidence verification -> human review`.

두 경로 모두 승격 게이트를 우회하지 않으며, 실제 배포는 사람 검토 이후에만 이뤄집니다.